# Асинхронный веб-краулер

### Установка зависимостей

In [24]:
%pip install aiohttp aiofiles beautifulsoup4 lxml aiosqlite pyyaml

Note: you may need to restart the kernel to use updated packages.


### Импорты
Все библиотеки, которые используются в ноутбуке, собраны здесь.

In [2]:
import asyncio
import aiohttp
import logging
import time
import json
import itertools
import re
import aiofiles
import random
import aiosqlite
import textwrap
import csv
import io
import tempfile

from pathlib import Path
from collections import Counter, deque
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from contextlib import asynccontextmanager, suppress
from urllib.robotparser import RobotFileParser
from abc import ABC, abstractmethod
from datetime import datetime


### Настройка логирования

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("crawler")

   ## Классы краулера

In [4]:
from crawler import *

## Демонстрация 1 дня

In [5]:
DEMO_URLS = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/get",
    "https://httpbin.org/html",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/3",
    "https://httpbin.org/status/404",
]


async def run_sequential(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    results = {}
    start = time.perf_counter()
    try:
        for url in urls:
            page = await crawler.fetch_url(url)
            if page is not None:
                results[url] = page
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


async def run_parallel(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


def print_report(title: str, urls: list[str], results: dict[str, str], elapsed: float):
    print(f"\n{title}")
    for url in urls:
        mark = "✅" if url in results else "❌"
        print(f"  {mark} {url}")
    print(f"  Загружено {len(results)} из {len(urls)} за {elapsed:.2f} сек")


async def main():
    seq_results, seq_time = await run_sequential(DEMO_URLS)
    par_results, par_time = await run_parallel(DEMO_URLS)

    print_report("Последовательная загрузка", DEMO_URLS, seq_results, seq_time)
    print_report("Параллельная загрузка", DEMO_URLS, par_results, par_time)
    print(f"\nПараллельно быстрее в {seq_time / par_time:.1f} раза")


await main()

22:45:28 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://example.com
22:45:28 [INFO] Попытка 1 успешна: https://example.com (статус 200, 577 символов)
22:45:28 [INFO] Итог: успех с попытки 1: https://example.com
22:45:28 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://www.python.org
22:45:29 [INFO] Попытка 1 успешна: https://www.python.org (статус 200, 52826 символов)
22:45:29 [INFO] Итог: успех с попытки 1: https://www.python.org
22:45:29 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://httpbin.org/get
22:45:30 [INFO] Попытка 1 успешна: https://httpbin.org/get (статус 200, 312 символов)
22:45:30 [INFO] Итог: успех с попытки 1: https://httpbin.org/get
22:45:30 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://httpbin.org/html
22:45:30 [INFO] Попытка 1 успешна: https://httpbin.org/html (статус 200, 3739 символов)
22:45:30 [INFO] Итог: успех с попытки 1: https://httpbin.org/html
22:45:30 [INFO] Начало загруз


Последовательная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 9.29 сек

Параллельная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 4.45 сек

Параллельно быстрее в 2.1 раза


## Тестирование 1 дня

In [6]:
async def test_valid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://example.com",
            "https://httpbin.org/html",
        ])
    finally:
        await crawler.close()
    assert len(results) == 2, f"ожидали 2 страницы, получили {len(results)}"
    assert "Example Domain" in results["https://example.com"], "не тот текст страницы"


async def test_invalid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://this-site-does-not-exist-12345.com",
            "https://httpbin.org/status/404",
        ])
    finally:
        await crawler.close()
    assert results == {}, f"ожидали пустой результат, получили {list(results)}"


async def test_timeout():
    crawler = AsyncCrawler(read_timeout=1)
    start = time.perf_counter()
    try:
        page = await crawler.fetch_url("https://httpbin.org/delay/5")
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start
    assert page is None, "страница не должна была загрузиться"
    assert elapsed < 4, f"таймаут не сработал, ждали {elapsed:.1f} сек"


async def test_parallel_faster():
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/delay/3",
    ]
    _, seq_time = await run_sequential(urls)
    _, par_time = await run_parallel(urls)
    assert par_time < seq_time * 0.75, (
        f"параллельно {par_time:.1f} сек, последовательно {seq_time:.1f} сек"
    )


async def run_tests(tests: list):
    logger.setLevel(logging.WARNING)
    passed = 0
    try:
        for test in tests:
            try:
                await test()
                print(f"✅ {test.__name__}")
                passed += 1
            except AssertionError as e:
                print(f"❌ {test.__name__}: {e}")
    finally:
        logger.setLevel(logging.INFO)
    print(f"\nПройдено {passed} из {len(tests)}")


await run_tests([test_valid_urls, test_invalid_urls, test_timeout, test_parallel_faster])

22:45:42 [ERROR] Попытка 1: NetworkError (ClientConnectorDNSError): https://this-site-does-not-exist-12345.com
22:45:42 [ERROR] Итог: неудача после 1 попыток (NetworkError: ClientConnectorDNSError): https://this-site-does-not-exist-12345.com


✅ test_valid_urls


22:45:43 [ERROR] Попытка 1: PermanentError (HTTP 404): https://httpbin.org/status/404
22:45:43 [ERROR] Итог: неудача после 1 попыток (PermanentError: HTTP 404): https://httpbin.org/status/404


✅ test_invalid_urls


22:45:45 [ERROR] Попытка 1: TransientError (SocketTimeoutError): https://httpbin.org/delay/5
22:45:45 [ERROR] Итог: неудача после 1 попыток (TransientError: SocketTimeoutError): https://httpbin.org/delay/5


✅ test_timeout
✅ test_parallel_faster

Пройдено 4 из 4


## Демонстрация 2 дня

In [7]:
DEMO_PAGES = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/html",
    "https://docs.python.org/3/",
]


def summarize(page: dict) -> dict:
    return {
        "url": page["url"],
        "title": page["title"],
        "h1": page["headings"].get("h1", []),
        "text_length": len(page["text"]),
        "links_count": len(page["links"]),
        "links": page["links"][:5],
        "images_count": len(page["images"]),
    }


async def demo_parsing(urls: list[str]):
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        pages = await asyncio.gather(*(crawler.fetch_and_parse(url) for url in urls))
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start

    for page in pages:
        print(json.dumps(summarize(page), ensure_ascii=False, indent=2))

    total_links = sum(len(page["links"]) for page in pages)
    total_text = sum(len(page["text"]) for page in pages)
    print("\nСтатистика")
    print(f"  Страниц: {len(pages)}")
    print(f"  Всего ссылок: {total_links}")
    print(f"  Общая длина текста: {total_text} символов")
    print(f"  Время: {elapsed:.2f} сек")


await demo_parsing(DEMO_PAGES)

22:45:55 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://example.com
22:45:55 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://www.python.org
22:45:55 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://httpbin.org/html
22:45:55 [INFO] Начало загрузки (попытка 1, таймаут чтения 10.0 сек): https://docs.python.org/3/
22:45:56 [INFO] Попытка 1 успешна: https://example.com (статус 200, 577 символов)
22:45:56 [INFO] Итог: успех с попытки 1: https://example.com
22:45:56 [INFO] Попытка 1 успешна: https://docs.python.org/3/ (статус 200, 19620 символов)
22:45:56 [INFO] Итог: успех с попытки 1: https://docs.python.org/3/
22:45:56 [INFO] Попытка 1 успешна: https://www.python.org (статус 200, 52826 символов)
22:45:56 [INFO] Итог: успех с попытки 1: https://www.python.org
22:45:56 [INFO] Попытка 1 успешна: https://httpbin.org/html (статус 200, 3739 символов)
22:45:56 [INFO] Итог: успех с попытки 1: https://httpbin.org/html


{
  "url": "https://example.com",
  "title": "Example Domain",
  "h1": [],
  "text_length": 171,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://www.python.org",
  "title": "Welcome to Python.org",
  "h1": [
    "Functions Defined",
    "All the Flow You’d Expect",
    "Compound Data Types",
    "Intuitive Interpretation",
    "Quick & Easy to Learn"
  ],
  "text_length": 6654,
  "links_count": 213,
  "links": [
    "https://www.python.org#content",
    "https://www.python.org#python-network",
    "https://www.python.org/",
    "https://www.python.org/psf/",
    "https://docs.python.org"
  ],
  "images_count": 1
}
{
  "url": "https://httpbin.org/html",
  "title": "",
  "h1": [
    "Herman Melville - Moby-Dick"
  ],
  "text_length": 3594,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://docs.python.org/3/",
  "title": "3.15.1 Documentation",
  "h1": [
    "Python 3.15.1 documentation"
  ],
  "text_length": 3217,
  "links_count": 9

## Тестирование 2 дня

In [8]:
VALID_HTML = """
<html>
  <head>
    <title>Тестовая страница</title>
    <meta name="description" content="Описание страницы">
  </head>
  <body>
    <h1>Главный заголовок</h1>
    <p>Основной текст страницы.</p>
    <img src="/logo.png" alt="Логотип">
    <table><tr><th>Город</th></tr><tr><td>Москва</td></tr></table>
    <ul><li>Чай</li><li>Кофе</li></ul>
  </body>
</html>
"""


async def test_parse_valid_html():
    result = await HTMLParser().parse_html(VALID_HTML, "https://example.com")
    assert result["title"] == "Тестовая страница", f"title: {result['title']}"
    assert result["metadata"]["description"] == "Описание страницы", "нет description"
    assert result["headings"]["h1"] == ["Главный заголовок"], "неверный h1"
    assert "Основной текст страницы." in result["text"], "нет текста"
    assert result["images"] == [
        {"src": "https://example.com/logo.png", "alt": "Логотип"}
    ], "неверные картинки"
    assert result["tables"] == [[["Город"], ["Москва"]]], "неверная таблица"
    assert result["lists"] == [{"type": "ul", "items": ["Чай", "Кофе"]}], "неверный список"


async def test_parse_broken_html():
    parser = HTMLParser()
    broken = "<title>Битая</title><h1>Заголовок<p>Текст<a href='/page'>ссылка<table><tr><td>A<td>B"
    result = await parser.parse_html(broken, "https://example.com")
    assert result["title"] == "Битая", f"title: {result['title']}"
    assert result["links"] == ["https://example.com/page"], f"links: {result['links']}"
    assert result["tables"] == [[["A", "B"]]], f"tables: {result['tables']}"

    result = await parser.parse_html(None, "https://example.com")
    assert result == parser.empty_result("https://example.com"), "ожидали пустой результат"


async def test_extract_links():
    html = """
    <a href="https://python.org">1</a>
    <a href="mailto:info@example.com">2</a>
    <a href="javascript:void(0)">3</a>
    <a href="tel:+70000000000">4</a>
    <a>без href</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com")
    assert links == ["https://python.org"], f"links: {links}"


async def test_relative_urls():
    html = """
    <a href="/about">1</a>
    <a href="page2.html">2</a>
    <a href="../archive">3</a>
    <a href="//cdn.example.org/lib">4</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com/news/today")
    assert links == [
        "https://example.com/about",
        "https://example.com/news/page2.html",
        "https://example.com/archive",
        "https://cdn.example.org/lib",
    ], f"links: {links}"


await run_tests([test_parse_valid_html, test_parse_broken_html, test_extract_links, test_relative_urls])

22:45:56 [WARNING] Не удалось разобрать HTML (TypeError): https://example.com


✅ test_parse_valid_html
✅ test_parse_broken_html
✅ test_extract_links
✅ test_relative_urls

Пройдено 4 из 4


## Демонстрация 3 дня

In [9]:
async def save_results(crawler: AsyncCrawler, path: str):
    data = {
        "processed": [
            {**page, "depth": crawler.url_depths[url]}
            for url, page in crawler.processed_urls.items()
        ],
        "failed": crawler.failed_urls,
    }
    async with aiofiles.open(path, "w", encoding="utf-8") as file:
        await file.write(json.dumps(data, ensure_ascii=False, indent=2))


async def demo_crawl():
    crawler = AsyncCrawler(max_concurrent=10, max_depth=2, per_domain_limit=3)
    logger.setLevel(logging.WARNING)
    try:
        results = await crawler.crawl(
            start_urls=[
                "https://quotes.toscrape.com/",
                "https://books.toscrape.com/",
            ],
            max_pages=150,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()

    await save_results(crawler, "crawl_results.json")

    by_domain = Counter(urlparse(url).netloc for url in results)
    by_depth = Counter(crawler.url_depths[url] for url in results)
    print(f"\nОбработано: {len(results)} страниц")
    print(f"Ошибок: {len(crawler.failed_urls)}")
    print(f"По сайтам: {dict(by_domain)}")
    print(f"По глубине: {dict(sorted(by_depth.items()))}")
    print("Результаты сохранены в crawl_results.json")


await demo_crawl()

[░░░░░░░░░░░░░░░░░░░░]   4% (2/48) | Скорость: 2.0 стр/сек | Осталось: ~23 сек | Активных задач: 10 | В очереди: 36 | Ошибок: 0 | Заблокировано: 0 | Запросов: 6.0 req/sec | Средняя задержка: 0.00 сек
[█░░░░░░░░░░░░░░░░░░░]   6% (7/123) | Скорость: 3.5 стр/сек | Осталось: ~33 сек | Активных задач: 10 | В очереди: 106 | Ошибок: 0 | Заблокировано: 0 | Запросов: 5.0 req/sec | Средняя задержка: 0.00 сек
[██░░░░░░░░░░░░░░░░░░]  12% (18/150) | Скорость: 6.0 стр/сек | Осталось: ~22 сек | Активных задач: 10 | В очереди: 148 | Ошибок: 0 | Заблокировано: 0 | Запросов: 7.0 req/sec | Средняя задержка: 0.00 сек
[████░░░░░░░░░░░░░░░░]  20% (30/150) | Скорость: 7.5 стр/сек | Осталось: ~16 сек | Активных задач: 10 | В очереди: 162 | Ошибок: 0 | Заблокировано: 0 | Запросов: 8.2 req/sec | Средняя задержка: 0.00 сек
[█████░░░░░░░░░░░░░░░]  27% (40/150) | Скорость: 8.0 стр/сек | Осталось: ~14 сек | Активных задач: 10 | В очереди: 163 | Ошибок: 0 | Заблокировано: 0 | Запросов: 8.6 req/sec | Средняя задержка

## Тестирование 3 дня

In [10]:
FAKE_SITE = {
    "https://site.com/": '<a href="/a">a</a><a href="/b">b</a><a href="https://other.com/x">x</a>',
    "https://site.com/a": '<a href="/a/deep">deep</a><a href="/">home</a><a href="/b">b</a>',
    "https://site.com/b": '<a href="/a">a</a><a href="/login">login</a>',
    "https://site.com/a/deep": '<a href="/a/deeper">deeper</a>',
    "https://site.com/a/deeper": "",
    "https://site.com/login": "",
    "https://other.com/x": "",
}


class FakeCrawler(AsyncCrawler):
    def __init__(self, site: dict | None = None, delays: dict | None = None, **kwargs):
        super().__init__(**kwargs)
        self.site = site or FAKE_SITE
        self.delays = delays or {}
        self.fetch_calls: list[str] = []

    async def fetch_url(self, url: str) -> str | None:
        self.fetch_calls.append(url)
        await asyncio.sleep(self.delays.get(url, 0.01))
        if url in self.site:
            return f"<html><body>{self.site[url]}</body></html>"
        self.failed_urls[url] = "NotFound"
        return None


async def run_fake_crawl(
    max_depth: int, site: dict | None = None, delays: dict | None = None, **filters
) -> FakeCrawler:
    crawler = FakeCrawler(site=site, delays=delays, max_concurrent=3, max_depth=max_depth)
    try:
        await crawler.crawl(["https://site.com/"], max_pages=100, **filters)
    finally:
        await crawler.close()
    return crawler


async def test_queue_priority():
    queue = CrawlerQueue()
    queue.add_url("https://site.com/a", priority=0)
    queue.add_url("https://site.com/b", priority=5)
    queue.add_url("https://site.com/c", priority=0)
    queue.add_url("https://site.com/d", priority=-1)
    queue.add_url("https://site.com/b", priority=10)

    order = [await queue.get_next() for _ in range(5)]
    assert order == [
        "https://site.com/b",
        "https://site.com/a",
        "https://site.com/c",
        "https://site.com/d",
        None,
    ], f"порядок: {order}"


async def test_depth_limit():
    crawler = await run_fake_crawl(max_depth=1)
    depths = [crawler.url_depths[url] for url in crawler.visited_urls]
    assert max(depths) == 1, f"глубины: {depths}"
    assert "https://site.com/a/deep" not in crawler.visited_urls, "зашли глубже max_depth"

    crawler = await run_fake_crawl(max_depth=2)
    assert "https://site.com/a/deep" in crawler.visited_urls, "не дошли до глубины 2"
    assert "https://site.com/a/deeper" not in crawler.visited_urls, "зашли на глубину 3"


async def test_url_filters():
    crawler = await run_fake_crawl(
        max_depth=3, same_domain_only=True, exclude_patterns=[r"/login"]
    )
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/b",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"

    crawler = await run_fake_crawl(max_depth=3, include_patterns=[r"/a"])
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"


async def test_no_duplicates():
    crawler = await run_fake_crawl(max_depth=3)
    calls = crawler.fetch_calls
    assert len(calls) == len(set(calls)), f"повторные загрузки: {calls}"
    assert set(calls) == crawler.visited_urls, "visited_urls не совпадает с загрузками"
    assert len(crawler.visited_urls) == len(FAKE_SITE), f"посещено {len(crawler.visited_urls)}"

SHORTCUT_SITE = {
    "https://site.com/": '<a href="/slow">slow</a><a href="/fast1">fast1</a>',
    "https://site.com/slow": '<a href="/target">target</a>',
    "https://site.com/fast1": '<a href="/fast2">fast2</a>',
    "https://site.com/fast2": '<a href="/target">target</a>',
    "https://site.com/target": '<a href="/child">child</a>',
    "https://site.com/child": "",
}

async def test_shortest_depth():
    crawler = await run_fake_crawl(
        max_depth=3,
        site=SHORTCUT_SITE,
        delays={"https://site.com/slow": 0.5},
    )
    assert crawler.url_depths["https://site.com/target"] == 2, (
        f"глубина target: {crawler.url_depths['https://site.com/target']}"
    )
    assert "https://site.com/child" in crawler.visited_urls, "страница в пределах max_depth потеряна"

await run_tests([test_queue_priority, test_depth_limit, test_url_filters, test_no_duplicates, test_shortest_depth])


✅ test_queue_priority
[████████████████████] 100% (4/4) | Скорость: 31.6 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
[████████████████████] 100% (6/6) | Скорость: 43.4 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_depth_limit
[████████████████████] 100% (5/5) | Скорость: 33.0 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
[████████████████████] 100% (4/4) | Скорость: 27.2 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_url_filters
[████████████████████] 100% (7/7) | Скорость: 46.7 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Запрос

## Демонстрация 4 дня

In [11]:
async def demo_polite_crawl():
    crawler = AsyncCrawler(
        max_concurrent=5,
        max_depth=1,
        requests_per_second=2.0,
        respect_robots=True,
        min_delay=0.5,
        jitter=0.2,
        user_agent="MyBot/1.0",
    )
    logger.setLevel(logging.WARNING)
    start = time.perf_counter()
    try:
        results = await crawler.crawl(
            start_urls=[
                "https://docs.python.org/3/",
                "https://quotes.toscrape.com/",
            ],
            max_pages=50,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()
    elapsed = time.perf_counter() - start

    rate_stats = crawler.rate_limiter.get_stats()
    by_domain = Counter(urlparse(url).netloc for url in results)
    print("\nИтоги")
    print(f"  Обработано страниц: {len(results)}")
    print(f"  По сайтам: {dict(by_domain)}")
    print(f"  Заблокировано robots.txt: {len(crawler.blocked_urls)}")
    print(f"  Ошибок: {len(crawler.failed_urls)}")
    print(f"  Время: {elapsed:.1f} сек")

    print("\nЗадержки")
    print(f"  Разрешено не чаще чем раз в: {crawler.rate_limiter.interval:.2f} сек (плюс до {crawler.jitter:.2f} случайно)")
    print(f"  Самая короткая фактическая пауза: {rate_stats['min_delay']:.2f} сек")
    print(f"  Средняя фактическая пауза: {rate_stats['avg_delay']:.2f} сек")

    print("\nЗаблокированные адреса")
    for url in sorted(crawler.blocked_urls):
        print(f"  {url}")


await demo_polite_crawl()

[░░░░░░░░░░░░░░░░░░░░]   4% (2/46) | Скорость: 2.0 стр/сек | Осталось: ~22 сек | Активных задач: 1 | В очереди: 43 | Ошибок: 0 | Заблокировано: 0 | Запросов: 3.0 req/sec | Средняя задержка: 0.51 сек


22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.7/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.6/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.5/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.4/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.3/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.2/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.1/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.0/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/2.7/
22:46:13 [WARNING] Заблокировано robots.txt: https://docs.python.org/2.6/


[██░░░░░░░░░░░░░░░░░░]  10% (5/50) | Скорость: 2.5 стр/сек | Осталось: ~18 сек | Активных задач: 0 | В очереди: 87 | Ошибок: 0 | Заблокировано: 0 | Запросов: 3.0 req/sec | Средняя задержка: 0.56 сек
[██████░░░░░░░░░░░░░░]  32% (16/50) | Скорость: 5.3 стр/сек | Осталось: ~6 сек | Активных задач: 1 | В очереди: 75 | Ошибок: 0 | Заблокировано: 10 | Запросов: 2.6 req/sec | Средняя задержка: 0.66 сек
[███████░░░░░░░░░░░░░]  38% (19/50) | Скорость: 4.7 стр/сек | Осталось: ~7 сек | Активных задач: 1 | В очереди: 72 | Ошибок: 0 | Заблокировано: 10 | Запросов: 2.7 req/sec | Средняя задержка: 0.79 сек
[████████░░░░░░░░░░░░]  44% (22/50) | Скорость: 4.4 стр/сек | Осталось: ~6 сек | Активных задач: 0 | В очереди: 70 | Ошибок: 0 | Заблокировано: 10 | Запросов: 2.6 req/sec | Средняя задержка: 0.80 сек
[█████████░░░░░░░░░░░]  48% (24/50) | Скорость: 4.0 стр/сек | Осталось: ~7 сек | Активных задач: 1 | В очереди: 67 | Ошибок: 0 | Заблокировано: 10 | Запросов: 2.4 req/sec | Средняя задержка: 0.79 сек
[

22:46:25 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.8/


[████████████████████] 100% (50/50) | Скорость: 3.4 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 42 | Ошибок: 0 | Заблокировано: 11 | Запросов: 3.0 req/sec | Средняя задержка: 0.72 сек

Итоги
  Обработано страниц: 39
  По сайтам: {'docs.python.org': 20, 'quotes.toscrape.com': 19}
  Заблокировано robots.txt: 11
  Ошибок: 0
  Время: 14.6 сек

Задержки
  Разрешено не чаще чем раз в: 0.50 сек (плюс до 0.20 случайно)
  Самая короткая фактическая пауза: 0.51 сек
  Средняя фактическая пауза: 0.72 сек

Заблокированные адреса
  https://docs.python.org/2.6/
  https://docs.python.org/2.7/
  https://docs.python.org/3.0/
  https://docs.python.org/3.1/
  https://docs.python.org/3.2/
  https://docs.python.org/3.3/
  https://docs.python.org/3.4/
  https://docs.python.org/3.5/
  https://docs.python.org/3.6/
  https://docs.python.org/3.7/
  https://docs.python.org/3.8/


## Тестирование 4 дня

In [12]:
class FakeResponse:
    def __init__(self, status: int, text: str):
        self.status = status
        self._text = text

    async def text(self) -> str:
        return self._text

    async def __aenter__(self):
        return self

    async def __aexit__(self, *args):
        return False


class FakeSession:
    def __init__(self, pages: dict[str, str]):
        self.pages = pages
        self.calls: list[str] = []

    def get(self, url: str) -> FakeResponse:
        self.calls.append(url)
        if url in self.pages:
            return FakeResponse(200, self.pages[url])
        return FakeResponse(404, "")


class FakePoliteCrawler(AsyncCrawler):
    def __init__(self, site: dict[str, str], robots_pages: dict[str, str], **kwargs):
        super().__init__(**kwargs)
        self.site = site
        self.robots = RobotsParser(FakeSession(robots_pages))
        self.request_times: list[float] = []

    async def try_fetch(self, url: str) -> str:
        async with self.semaphores.acquire(url):
            await self.rate_limiter.acquire(urlparse(url).netloc)
            self.request_times.append(time.monotonic())
            if url in self.site:
                return f"<html><body>{self.site[url]}</body></html>"
            raise PermanentError("NotFound", url, 404)


def gaps_between(times: list[float]) -> list[float]:
    times = sorted(times)
    return [later - earlier for earlier, later in zip(times, times[1:])]


async def test_rate_limit_single_domain():
    limiter = RateLimiter(requests_per_second=10.0)
    times = []

    async def request():
        await limiter.acquire("a.com")
        times.append(time.monotonic())

    await asyncio.gather(*(request() for _ in range(5)))
    gaps = gaps_between(times)
    assert all(gap >= 0.09 for gap in gaps), f"паузы: {gaps}"


async def test_rate_limit_multiple_domains():
    domains = ["a.com", "a.com", "a.com", "b.com", "b.com", "b.com"]
    for per_domain, expected in ((True, 0.2), (False, 0.5)):
        limiter = RateLimiter(requests_per_second=10.0, per_domain=per_domain)
        start = time.monotonic()
        await asyncio.gather(*(limiter.acquire(domain) for domain in domains))
        elapsed = time.monotonic() - start
        assert expected - 0.02 <= elapsed <= expected + 0.05, (
            f"per_domain={per_domain}: {elapsed:.2f} сек вместо {expected}"
        )


ROBOTS_TXT = """
User-agent: *
Disallow: /private/
Crawl-delay: 2

User-agent: BadBot
Disallow: /

User-agent: SlowBot
Crawl-delay: 0.5
"""


async def test_robots_parsing():
    session = FakeSession({"https://site.com/robots.txt": ROBOTS_TXT})
    robots = RobotsParser(session)
    await robots.fetch_robots("https://site.com/page")
    info = await robots.fetch_robots("https://site.com/other")
    assert session.calls == ["https://site.com/robots.txt"], f"загрузок: {session.calls}"
    assert info["crawl_delay"] == 2.0, f"crawl_delay: {info['crawl_delay']}"
    assert robots.get_crawl_delay("SlowBot/1.0", "site.com") == 0.5, "дробный Crawl-delay не прочитан"
    assert robots.can_fetch("https://site.com/public") is True, "public должен быть разрешён"
    assert robots.can_fetch("https://site.com/private/data") is False, "private должен быть запрещён"
    assert robots.can_fetch("https://site.com/public", "BadBot") is False, "BadBot должен быть заблокирован"

    info = await robots.fetch_robots("https://empty.com/")
    assert info["allow_all"] is True, "без robots.txt всё разрешено"


POLITE_SITE = {
    "https://polite.com/": '<a href="/public">public</a><a href="/private/secret">secret</a>',
    "https://polite.com/public": "",
    "https://polite.com/private/secret": "",
}


async def test_blocked_urls():
    crawler = FakePoliteCrawler(
        site=POLITE_SITE,
        robots_pages={"https://polite.com/robots.txt": "User-agent: *\nDisallow: /private/\n"},
        respect_robots=True,
        max_depth=1,
    )
    try:
        results = await crawler.crawl(["https://polite.com/"])
    finally:
        await crawler.close()
    assert crawler.blocked_urls == {"https://polite.com/private/secret"}, f"blocked: {crawler.blocked_urls}"
    assert set(results) == {"https://polite.com/", "https://polite.com/public"}, f"results: {set(results)}"
    assert len(crawler.request_times) == 2, f"запросов: {len(crawler.request_times)}"


async def test_delays_respected():
    limiter = RateLimiter(requests_per_second=100.0, min_delay=0.2, jitter=0.1)
    times = []
    for _ in range(4):
        await limiter.acquire("a.com")
        times.append(time.monotonic())
    gaps = gaps_between(times)
    assert all(0.19 <= gap <= 0.32 for gap in gaps), f"min_delay и jitter: {gaps}"

    site = {f"https://polite.com/page{i}": "" for i in range(4)}
    crawler = FakePoliteCrawler(
        site=site,
        robots_pages={"https://polite.com/robots.txt": "User-agent: *\nCrawl-delay: 0.5\n"},
        respect_robots=True,
    )
    try:
        await crawler.fetch_urls(list(site))
    finally:
        await crawler.close()
    gaps = gaps_between(crawler.request_times)
    assert all(gap >= 0.49 for gap in gaps), f"Crawl-delay: {gaps}"


def make_site(domain: str, pages: int) -> dict[str, str]:
    links = "".join(f'<a href="/page{i}">{i}</a>' for i in range(pages))
    site = {f"https://{domain}/": links}
    site.update({f"https://{domain}/page{i}": "" for i in range(pages)})
    return site


async def test_domains_in_parallel():
    site = {**make_site("a.com", 6), **make_site("b.com", 6)}
    crawler = FakePoliteCrawler(
        site=site,
        robots_pages={},
        requests_per_second=2.0,
        max_concurrent=3,
        max_depth=1,
    )
    start = time.monotonic()
    try:
        results = await crawler.crawl(["https://a.com/", "https://b.com/"])
    finally:
        await crawler.close()
    elapsed = time.monotonic() - start
    assert len(results) == 14, f"обработано {len(results)}"
    assert elapsed < 4.0, f"сайты обходились по очереди: {elapsed:.1f} сек"


await run_tests([
    test_rate_limit_single_domain,
    test_rate_limit_multiple_domains,
    test_robots_parsing,
    test_blocked_urls,
    test_delays_respected,
    test_domains_in_parallel,
])

✅ test_rate_limit_single_domain


22:46:26 [WARNING] Заблокировано robots.txt: https://polite.com/private/secret


✅ test_rate_limit_multiple_domains
✅ test_robots_parsing
[████████████████████] 100% (3/3) | Скорость: 29.2 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 0 | Заблокировано: 1 | Запросов: 2.0 req/sec | Средняя задержка: 0.00 сек
✅ test_blocked_urls
✅ test_delays_respected
[█████░░░░░░░░░░░░░░░]  29% (4/14) | Скорость: 4.0 стр/сек | Осталось: ~3 сек | Активных задач: 0 | В очереди: 10 | Ошибок: 0 | Заблокировано: 0 | Запросов: 4.0 req/sec | Средняя задержка: 0.50 сек
[███████████░░░░░░░░░]  57% (8/14) | Скорость: 4.0 стр/сек | Осталось: ~2 сек | Активных задач: 0 | В очереди: 6 | Ошибок: 0 | Заблокировано: 0 | Запросов: 4.0 req/sec | Средняя задержка: 0.51 сек
[█████████████████░░░]  86% (12/14) | Скорость: 4.0 стр/сек | Осталось: ~1 сек | Активных задач: 0 | В очереди: 2 | Ошибок: 0 | Заблокировано: 0 | Запросов: 4.0 req/sec | Средняя задержка: 0.51 сек
[████████████████████] 100% (14/14) | Скорость: 4.4 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очеред

## Демонстрация 5 дня

In [13]:
async def save_error_report(crawler: AsyncCrawler, path: str):
    report = {
        "statistics": crawler.get_error_stats(),
        "failed_urls": crawler.failed_urls,
        "circuit_breaker": {
            domain: crawler.circuit_breaker.get_state(domain)
            for domain in crawler.circuit_breaker.results
        },
    }
    async with aiofiles.open(path, "w", encoding="utf-8") as file:
        await file.write(json.dumps(report, ensure_ascii=False, indent=2))


async def demo_error_handling():
    strategy = RetryStrategy(
        max_retries=3,
        backoff_factor=2.0,
        retry_on=[TransientError, NetworkError],
        base_delay=0.5,
        retry_limits={NetworkError: 1},
    )
    breaker = CircuitBreaker(failure_rate=0.9, window_size=10, min_requests=4, recovery_timeout=30.0)
    crawler = AsyncCrawler(
        max_concurrent=5,
        read_timeout=1,
        timeout_growth=2.0,
        retry_strategy=strategy,
        circuit_breaker=breaker,
    )
    urls = [
        "https://httpbin.org/get",
        "https://httpbin.org/html",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/status/429",
        "https://httpbin.org/status/500",
        "https://httpbin.org/status/404",
        "https://httpbin.org/status/403",
        "https://this-site-does-not-exist-12345.com",
        "https://httpstat.us/503",
        "https://httpstat.us/503?n=1",
        "https://httpstat.us/503?n=2",
        "https://httpstat.us/503?n=3",
    ]
    logger.setLevel(logging.WARNING)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()
    elapsed = time.perf_counter() - start

    await save_error_report(crawler, "error_report.json")

    stats = crawler.get_error_stats()
    print("\nИтоги")
    print(f"  Загружено: {len(results)} из {len(urls)} за {elapsed:.1f} сек")
    print(f"  Ошибки по типам: {stats['errors_by_type']}")
    print(f"  Всего повторов: {stats['retries']}, из них спасли запрос: {stats['successful_retries']}")
    print(f"  Среднее время на повтор: {stats['avg_retry_time']:.2f} сек")
    print("  Постоянные ошибки:")
    for url, reason in stats["permanent_urls"].items():
        print(f"    {url}: {reason}")
    print("Отчёт сохранён в error_report.json")


await demo_error_handling()

22:46:33 [ERROR] Попытка 1: NetworkError (ClientConnectorDNSError): https://this-site-does-not-exist-12345.com
22:46:33 [WARNING] NetworkError (ClientConnectorDNSError) на попытке 1, повтор через 0.50 сек: https://this-site-does-not-exist-12345.com
22:46:34 [ERROR] Попытка 1: TransientError (HTTP 429): https://httpbin.org/status/429
22:46:34 [WARNING] TransientError (HTTP 429) на попытке 1, повтор через 1.50 сек: https://httpbin.org/status/429
22:46:34 [ERROR] Попытка 1: TransientError (HTTP 500): https://httpbin.org/status/500
22:46:34 [WARNING] TransientError (HTTP 500) на попытке 1, повтор через 0.50 сек: https://httpbin.org/status/500
22:46:34 [ERROR] Попытка 1: TransientError (SocketTimeoutError): https://httpbin.org/delay/2
22:46:34 [WARNING] TransientError (SocketTimeoutError) на попытке 1, повтор через 0.50 сек: https://httpbin.org/delay/2
22:46:35 [ERROR] Попытка 1: PermanentError (HTTP 404): https://httpbin.org/status/404
22:46:35 [ERROR] Итог: неудача после 1 попыток (Perman


Итоги
  Загружено: 3 из 12 за 12.9 сек
  Ошибки по типам: {'NetworkError': 2, 'TransientError': 13, 'PermanentError': 2, 'CircuitOpenError': 4}
  Всего повторов: 12, из них спасли запрос: 1
  Среднее время на повтор: 2.07 сек
  Постоянные ошибки:
    https://httpbin.org/status/404: HTTP 404
    https://httpbin.org/status/403: HTTP 403
Отчёт сохранён в error_report.json


## Тестирование 5 дня

In [14]:
def http_error(status: int) -> aiohttp.ClientResponseError:
    return aiohttp.ClientResponseError(request_info=None, history=(), status=status)


class ScriptedResponse:
    def __init__(self, outcome):
        self.outcome = outcome
        self.status = outcome if isinstance(outcome, int) else 200
        self.headers = {"Content-Type": "text/html; charset=utf-8"}

    async def __aenter__(self):
        if isinstance(self.outcome, Exception):
            raise self.outcome
        return self

    async def __aexit__(self, *args):
        return False

    def raise_for_status(self):
        if self.status >= 400:
            raise http_error(self.status)

    async def text(self) -> str:
        return "<html><body>ok</body></html>"


class ScriptedSession:
    def __init__(self, script: dict[str, list]):
        self.script = {url: list(outcomes) for url, outcomes in script.items()}
        self.calls: list[tuple[str, float]] = []

    def get(self, url: str, **kwargs) -> ScriptedResponse:
        self.calls.append((url, time.monotonic()))
        outcomes = self.script.get(url, [])
        outcome = outcomes.pop(0) if outcomes else 200
        return ScriptedResponse(outcome)

    def calls_for(self, url: str) -> list[float]:
        return [moment for called_url, moment in self.calls if called_url == url]


class ScriptedCrawler(AsyncCrawler):
    def __init__(self, script: dict[str, list], **kwargs):
        super().__init__(**kwargs)
        self.real_session = self.session
        self.session = ScriptedSession(script)

    async def close(self):
        await self.real_session.close()


def fast_strategy(**kwargs) -> RetryStrategy:
    return RetryStrategy(max_retries=3, base_delay=0.01, **kwargs)


async def run_scripted(script: dict[str, list], strategy: RetryStrategy) -> ScriptedCrawler:
    crawler = ScriptedCrawler(script, retry_strategy=strategy)
    try:
        await crawler.fetch_urls(list(script))
    finally:
        await crawler.close()
    return crawler


async def test_retry_on_timeout():
    url = "https://site.com/slow"
    crawler = await run_scripted(
        {url: [asyncio.TimeoutError(), asyncio.TimeoutError(), 200]}, fast_strategy()
    )
    attempts = len(crawler.session.calls_for(url))
    assert attempts == 3, f"попыток: {attempts}"
    assert url not in crawler.failed_urls, "страница должна загрузиться с третьей попытки"


async def test_retry_on_503():
    ok_url = "https://site.com/recovers"
    bad_url = "https://site.com/always-503"
    crawler = await run_scripted({ok_url: [503, 503, 200], bad_url: [503] * 10}, fast_strategy())
    assert len(crawler.session.calls_for(ok_url)) == 3, "503 должен повторяться до успеха"
    assert ok_url not in crawler.failed_urls, "страница должна загрузиться"
    attempts = len(crawler.session.calls_for(bad_url))
    assert attempts == 4, f"попыток: {attempts}"
    assert crawler.failed_urls[bad_url] == "TransientError: HTTP 503", crawler.failed_urls[bad_url]


async def test_no_retry_on_404():
    url = "https://site.com/missing"
    crawler = await run_scripted({url: [404] * 5}, fast_strategy())
    attempts = len(crawler.session.calls_for(url))
    assert attempts == 1, f"попыток: {attempts}"
    assert crawler.failed_urls[url] == "PermanentError: HTTP 404", crawler.failed_urls[url]


async def test_exponential_backoff():
    strategy = RetryStrategy(max_retries=3, backoff_factor=2.0, base_delay=0.05)
    delays = [round(strategy.get_delay(n), 6) for n in (1, 2, 3)]
    assert delays == [0.05, 0.1, 0.2], f"паузы по формуле: {delays}"
    assert round(strategy.get_delay(1, http_error(429)), 6) == 0.15, "для 429 пауза втрое длиннее"

    url = "https://site.com/flaky"
    crawler = await run_scripted({url: [503, 503, 503, 200]}, strategy)
    times = crawler.session.calls_for(url)
    gaps = [later - earlier for earlier, later in zip(times, times[1:])]
    for gap, expected in zip(gaps, [0.05, 0.1, 0.2]):
        assert expected <= gap < expected + 0.05, f"фактические паузы: {gaps}"


async def test_error_stats():
    script = {
        "https://site.com/ok": [200],
        "https://site.com/saved": [503, 200],
        "https://site.com/missing": [404],
        "https://site.com/forbidden": [403],
        "https://site.com/dead": [aiohttp.ClientConnectionError()] * 5,
    }
    crawler = ScriptedCrawler(script, retry_strategy=fast_strategy(retry_limits={NetworkError: 1}))
    try:
        await crawler.fetch_urls(list(script))
        await crawler.parser.parse_html(None, "https://site.com/broken")
    finally:
        await crawler.close()

    stats = crawler.get_error_stats()
    assert stats["errors_by_type"] == {
        "TransientError": 1,
        "PermanentError": 2,
        "NetworkError": 2,
        "ParseError": 1,
    }, stats["errors_by_type"]
    assert stats["retries"] == 2, f"повторов: {stats['retries']}"
    assert stats["successful_retries"] == 1, f"успешных повторов: {stats['successful_retries']}"
    assert stats["avg_retry_time"] > 0, "среднее время на повтор не посчитано"
    assert set(stats["permanent_urls"]) == {
        "https://site.com/missing",
        "https://site.com/forbidden",
    }, stats["permanent_urls"]


async def test_circuit_breaker():
    breaker = CircuitBreaker(failure_rate=0.5, window_size=4, min_requests=2, recovery_timeout=0.2)
    breaker.record_failure("site.com")
    breaker.record_failure("site.com")
    assert breaker.get_state("site.com") == "open", "домен должен заблокироваться"
    assert breaker.allow_request("site.com") is False, "запросы к заблокированному домену запрещены"

    breaker.record_success("site.com")
    assert breaker.get_state("site.com") == "open", "успех без пробы не должен снимать блокировку"

    await asyncio.sleep(0.25)
    assert breaker.allow_request("site.com") is True, "после паузы должна пройти проба"
    assert breaker.allow_request("site.com") is False, "одновременно проходит только одна проба"
    breaker.record_success("site.com")
    assert breaker.get_state("site.com") == "closed", "успешная проба снимает блокировку"


await run_tests([
    test_retry_on_timeout,
    test_retry_on_503,
    test_no_retry_on_404,
    test_exponential_backoff,
    test_error_stats,
    test_circuit_breaker,
])

22:46:45 [ERROR] Попытка 1: TransientError (TimeoutError): https://site.com/slow
22:46:45 [WARNING] TransientError (TimeoutError) на попытке 1, повтор через 0.01 сек: https://site.com/slow
22:46:45 [ERROR] Попытка 2: TransientError (TimeoutError): https://site.com/slow
22:46:45 [WARNING] TransientError (TimeoutError) на попытке 2, повтор через 0.02 сек: https://site.com/slow
22:46:45 [ERROR] Попытка 1: TransientError (HTTP 503): https://site.com/recovers
22:46:45 [WARNING] TransientError (HTTP 503) на попытке 1, повтор через 0.01 сек: https://site.com/recovers
22:46:45 [ERROR] Попытка 1: TransientError (HTTP 503): https://site.com/always-503
22:46:45 [WARNING] TransientError (HTTP 503) на попытке 1, повтор через 0.01 сек: https://site.com/always-503
22:46:45 [ERROR] Попытка 2: TransientError (HTTP 503): https://site.com/recovers
22:46:45 [WARNING] TransientError (HTTP 503) на попытке 2, повтор через 0.02 сек: https://site.com/recovers
22:46:45 [ERROR] Попытка 2: TransientError (HTTP 50

✅ test_retry_on_timeout
✅ test_retry_on_503
✅ test_no_retry_on_404


22:46:45 [ERROR] Попытка 3: TransientError (HTTP 503): https://site.com/flaky
22:46:45 [WARNING] TransientError (HTTP 503) на попытке 3, повтор через 0.20 сек: https://site.com/flaky
22:46:45 [ERROR] Попытка 1: TransientError (HTTP 503): https://site.com/saved
22:46:45 [WARNING] TransientError (HTTP 503) на попытке 1, повтор через 0.01 сек: https://site.com/saved
22:46:45 [ERROR] Попытка 1: PermanentError (HTTP 404): https://site.com/missing
22:46:45 [ERROR] Итог: неудача после 1 попыток (PermanentError: HTTP 404): https://site.com/missing
22:46:45 [ERROR] Попытка 1: PermanentError (HTTP 403): https://site.com/forbidden
22:46:45 [ERROR] Итог: неудача после 1 попыток (PermanentError: HTTP 403): https://site.com/forbidden
22:46:45 [ERROR] Попытка 1: NetworkError (ClientConnectionError): https://site.com/dead
22:46:45 [WARNING] NetworkError (ClientConnectionError) на попытке 1, повтор через 0.01 сек: https://site.com/dead
22:46:45 [ERROR] Попытка 2: NetworkError (ClientConnectionError): h

✅ test_exponential_backoff
✅ test_error_stats
✅ test_circuit_breaker

Пройдено 6 из 6


## Демонстрация 6 дня

In [15]:
class MultiStorage(DataStorage):
    def __init__(self, *storages: DataStorage):
        self.storages = storages

    async def save(self, data: dict):
        await asyncio.gather(*(storage.save(data) for storage in self.storages))

    async def close(self):
        await asyncio.gather(*(storage.close() for storage in self.storages))


async def read_saved_data():
    print("\nЧтение JSON")
    with open("results.json", encoding="utf-8") as file:
        pages = json.load(file)
    print(f"  Страниц: {len(pages)}, первая: {pages[0]['title']}")

    print("\nЧтение CSV")
    with open("results.csv", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    print(f"  Строк: {len(rows)}, колонки: {list(rows[0])}")

    print("\nЧтение SQLite")
    async with aiosqlite.connect("crawler.db") as db:
        async with db.execute(
            "SELECT COUNT(*), AVG(LENGTH(text)), MIN(crawled_at), MAX(crawled_at) FROM pages"
        ) as cursor:
            count, avg_text, first_time, last_time = await cursor.fetchone()
        print(f"  Страниц: {count}, средняя длина текста: {avg_text:.0f} символов")
        print(f"  Обход шёл с {first_time} по {last_time}")

        async with db.execute(
            "SELECT status_code, COUNT(*) FROM pages GROUP BY status_code"
        ) as cursor:
            print(f"  По статусам: {dict(await cursor.fetchall())}")

        print("  Больше всего ссылок:")
        async with db.execute(
            "SELECT url, json_array_length(links) AS links_count FROM pages "
            "ORDER BY links_count DESC LIMIT 3"
        ) as cursor:
            async for url, links_count in cursor:
                print(f"    {links_count} ссылок: {url}")


async def demo_storage():
    storage = MultiStorage(
        JSONStorage("results.json", pretty=True),
        CSVStorage("results.csv"),
        SQLiteStorage("crawler.db"),
    )
    crawler = AsyncCrawler(
        max_concurrent=5,
        max_depth=1,
        requests_per_second=4.0,
        storage=storage,
    )
    logger.setLevel(logging.WARNING)
    try:
        results = await crawler.crawl(
            start_urls=["https://quotes.toscrape.com/"],
            max_pages=20,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()

    print("\nИтоги сохранения")
    print(f"  Обработано страниц: {len(results)}")
    print(f"  Сохранено: {crawler.saved_count}")
    print(f"  Ошибок сохранения: {len(crawler.save_errors)}")

    await read_saved_data()


await demo_storage()

[██░░░░░░░░░░░░░░░░░░]  10% (2/20) | Скорость: 2.0 стр/сек | Осталось: ~9 сек | Активных задач: 1 | В очереди: 44 | Ошибок: 0 | Заблокировано: 0 | Запросов: 3.0 req/sec | Средняя задержка: 0.25 сек
[███░░░░░░░░░░░░░░░░░]  15% (3/20) | Скорость: 1.5 стр/сек | Осталось: ~11 сек | Активных задач: 4 | В очереди: 40 | Ошибок: 0 | Заблокировано: 0 | Запросов: 3.5 req/sec | Средняя задержка: 0.25 сек
[██████████░░░░░░░░░░]  50% (10/20) | Скорость: 3.3 стр/сек | Осталось: ~3 сек | Активных задач: 1 | В очереди: 36 | Ошибок: 0 | Заблокировано: 0 | Запросов: 3.7 req/sec | Средняя задержка: 0.25 сек
[██████████████░░░░░░]  70% (14/20) | Скорость: 3.5 стр/сек | Осталось: ~2 сек | Активных задач: 1 | В очереди: 32 | Ошибок: 0 | Заблокировано: 0 | Запросов: 3.7 req/sec | Средняя задержка: 0.25 сек
[██████████████████░░]  90% (18/20) | Скорость: 3.6 стр/сек | Осталось: ~1 сек | Активных задач: 1 | В очереди: 28 | Ошибок: 0 | Заблокировано: 0 | Запросов: 3.6 req/sec | Средняя задержка: 0.25 сек
[█████

## Тестирование 6 дня

In [16]:
def sample_records(count: int) -> list[dict]:
    return [
        {
            "url": f"https://example.com/{i}",
            "title": f"Страница {i}",
            "text": "текст",
            "links": [f"https://example.com/{i + 1}"],
            "metadata": {"description": "описание"},
            "status_code": 200,
            "content_type": "text/html",
            "images": ["лишнее поле"],
        }
        for i in range(count)
    ]


async def test_json_storage():
    records = sample_records(10)
    with tempfile.TemporaryDirectory() as folder:
        for name, pretty in (("pages.jsonl", False), ("pages.json", True)):
            path = Path(folder) / name
            storage = JSONStorage(str(path), pretty=pretty, buffer_size=3)
            await asyncio.gather(*(storage.save(record) for record in records))
            await storage.close()
            assert storage.writes == 4, f"{name}: обращений к диску {storage.writes}"

            text = path.read_text(encoding="utf-8")
            saved = json.loads(text) if pretty else [json.loads(line) for line in text.splitlines()]
            assert len(saved) == 10, f"{name}: записей {len(saved)}"
            assert {page["url"] for page in saved} == {r["url"] for r in records}, f"{name}: адреса не совпали"
            assert all(list(page) == DataStorage.FIELDS for page in saved), f"{name}: поля не стандартные"


async def test_csv_storage():
    record = {
        "url": "https://example.com/tricky",
        "title": 'Кавычки "вот", запятая и\nперенос строки',
        "links": ["https://a.com", "https://b.com"],
        "metadata": {"keywords": "a, b"},
    }
    with tempfile.TemporaryDirectory() as folder:
        path = Path(folder) / "pages.csv"
        storage = CSVStorage(str(path))
        await storage.save(record)
        await storage.close()
        with open(path, encoding="utf-8-sig", newline="") as file:
            reader = csv.DictReader(file)
            rows = list(reader)
            headers = reader.fieldnames

    assert headers == DataStorage.FIELDS, f"заголовки: {headers}"
    assert len(rows) == 1, f"строк: {len(rows)}"
    assert rows[0]["title"] == record["title"], f"заголовок испорчен: {rows[0]['title']!r}"
    assert json.loads(rows[0]["links"]) == record["links"], "ссылки испорчены"
    assert json.loads(rows[0]["metadata"]) == record["metadata"], "метаданные испорчены"


async def test_sqlite_storage():
    records = sample_records(7)
    with tempfile.TemporaryDirectory() as folder:
        path = str(Path(folder) / "pages.db")
        storage = SQLiteStorage(path, batch_size=3)
        for record in records:
            await storage.save(record)
        await storage.save({**records[0], "title": "Обновлённая"})
        await storage.close()
        assert storage.batches_written == 3, f"пачек: {storage.batches_written}"

        async with aiosqlite.connect(path) as db:
            async with db.execute("SELECT COUNT(*) FROM pages") as cursor:
                count = (await cursor.fetchone())[0]
            async with db.execute("SELECT title FROM pages WHERE url = ?", (records[0]["url"],)) as cursor:
                title = (await cursor.fetchone())[0]
            async with db.execute("SELECT name FROM sqlite_master WHERE type = 'index'") as cursor:
                indexes = {row[0] async for row in cursor}

    assert count == 7, f"строк: {count}, повторный url должен заменить старую строку"
    assert title == "Обновлённая", f"заголовок: {title}"
    assert {"idx_pages_status", "idx_pages_crawled_at"} <= indexes, f"индексы: {indexes}"


class FlakyStorage(DataStorage):
    def __init__(self, failures: int):
        self.failures = failures
        self.saved: list[dict] = []

    async def save(self, data: dict):
        if self.failures > 0:
            self.failures -= 1
            raise OSError("диск временно недоступен")
        self.saved.append(data)

    async def close(self):
        pass


async def test_save_errors():
    storage = FlakyStorage(failures=2)
    crawler = AsyncCrawler(storage=storage, save_retries=2, save_retry_delay=0.01)
    try:
        saved = await crawler.save_page({"url": "https://example.com/1"})
    finally:
        await crawler.close()
    assert saved is True, "должно сохраниться с третьей попытки"
    assert len(storage.saved) == 1, f"сохранено {len(storage.saved)}"

    broken = FlakyStorage(failures=1000)
    crawler = FakeCrawler(
        max_concurrent=3, max_depth=3, storage=broken, save_retries=1, save_retry_delay=0.01
    )
    try:
        results = await crawler.crawl(["https://site.com/"])
    finally:
        await crawler.close()
    assert results == {}, f"несохранённые страницы попали в обработанные: {list(results)}"
    assert crawler.visited_urls == set(FAKE_SITE), f"обход прервался: посещено {len(crawler.visited_urls)}"
    assert len(crawler.save_errors) == len(FAKE_SITE), f"ошибок сохранения: {len(crawler.save_errors)}"
    assert all(error.startswith("SaveError") for error in crawler.failed_urls.values()), "нет пометки SaveError"


async def test_data_integrity():
    with tempfile.TemporaryDirectory() as folder:
        json_path = Path(folder) / "pages.json"
        csv_path = Path(folder) / "pages.csv"
        db_path = str(Path(folder) / "pages.db")
        storage = MultiStorage(
            JSONStorage(str(json_path), pretty=True),
            CSVStorage(str(csv_path)),
            SQLiteStorage(db_path),
        )
        crawler = FakeCrawler(max_concurrent=3, max_depth=3, storage=storage)
        try:
            results = await crawler.crawl(["https://site.com/"])
        finally:
            await crawler.close()

        from_json = {
            page["url"]: (page["title"], page["links"])
            for page in json.loads(json_path.read_text(encoding="utf-8"))
        }
        with open(csv_path, encoding="utf-8-sig", newline="") as file:
            from_csv = {
                row["url"]: (row["title"], json.loads(row["links"]))
                for row in csv.DictReader(file)
            }
        async with aiosqlite.connect(db_path) as db:
            async with db.execute("SELECT url, title, links FROM pages") as cursor:
                from_db = {url: (title, json.loads(links)) async for url, title, links in cursor}

    expected = {url: (page["title"], page["links"]) for url, page in results.items()}
    assert from_json == expected, "JSON не совпадает с результатами обхода"
    assert from_csv == expected, "CSV не совпадает с результатами обхода"
    assert from_db == expected, "SQLite не совпадает с результатами обхода"


await run_tests([
    test_json_storage,
    test_csv_storage,
    test_sqlite_storage,
    test_save_errors,
    test_data_integrity,
])

22:46:51 [WARNING] Ошибка сохранения на попытке 1 (OSError: диск временно недоступен), повтор через 0.01 сек: https://example.com/1
22:46:51 [WARNING] Ошибка сохранения на попытке 2 (OSError: диск временно недоступен), повтор через 0.02 сек: https://example.com/1
22:46:51 [WARNING] Ошибка сохранения на попытке 1 (OSError: диск временно недоступен), повтор через 0.01 сек: https://site.com/
22:46:51 [ERROR] Не удалось сохранить после 2 попыток (OSError: диск временно недоступен): https://site.com/
22:46:51 [WARNING] Ошибка сохранения на попытке 1 (OSError: диск временно недоступен), повтор через 0.01 сек: https://other.com/x
22:46:51 [WARNING] Ошибка сохранения на попытке 1 (OSError: диск временно недоступен), повтор через 0.01 сек: https://site.com/a
22:46:51 [WARNING] Ошибка сохранения на попытке 1 (OSError: диск временно недоступен), повтор через 0.01 сек: https://site.com/b
22:46:51 [ERROR] Не удалось сохранить после 2 попыток (OSError: диск временно недоступен): https://other.com/x


✅ test_json_storage
✅ test_csv_storage
✅ test_sqlite_storage


22:46:51 [ERROR] Не удалось сохранить после 2 попыток (OSError: диск временно недоступен): https://site.com/a/deeper


[████████████████████] 100% (7/7) | Скорость: 35.6 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 7 | Заблокировано: 0 | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_save_errors
[████████████████████] 100% (7/7) | Скорость: 45.8 стр/сек | Осталось: ~0 сек | Активных задач: 0 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_data_integrity

Пройдено 5 из 5


## Демонстрация 7 дня

In [17]:
from crawler import AdvancedCrawler


async def main():
    crawler = AdvancedCrawler.from_config("config.yaml")

    await crawler.crawl()

    stats = crawler.get_stats()
    print(f"Обработано: {stats['total_pages']} страниц")
    print(f"Успешно: {stats['successful']}")
    print(f"Ошибок: {stats['failed']}")

    crawler.export_to_html_report("report.html")
    await crawler.close()


await main()

22:46:52 [INFO] Конфигурация загружена из config.yaml


[░░░░░░░░░░░░░░░░░░░░]   3% (1/30) | Скорость: 1.0 стр/сек | Осталось: ~29 сек | Активных задач: 0 | В очереди: 45 | Ошибок: 0 | Заблокировано: 0 | Запросов: 1.0 req/sec | Средняя задержка: 0.00 сек
[█░░░░░░░░░░░░░░░░░░░]   7% (2/30) | Скорость: 1.0 стр/сек | Осталось: ~28 сек | Активных задач: 1 | В очереди: 63 | Ошибок: 0 | Заблокировано: 0 | Запросов: 2.1 req/sec | Средняя задержка: 0.58 сек
[███░░░░░░░░░░░░░░░░░]  17% (5/30) | Скорость: 1.7 стр/сек | Осталось: ~15 сек | Активных задач: 0 | В очереди: 77 | Ошибок: 0 | Заблокировано: 0 | Запросов: 2.0 req/sec | Средняя задержка: 0.56 сек
[████░░░░░░░░░░░░░░░░]  23% (7/30) | Скорость: 1.7 стр/сек | Осталось: ~13 сек | Активных задач: 0 | В очереди: 78 | Ошибок: 0 | Заблокировано: 0 | Запросов: 2.0 req/sec | Средняя задержка: 0.55 сек
[█████░░░░░░░░░░░░░░░]  27% (8/30) | Скорость: 1.6 стр/сек | Осталось: ~14 сек | Активных задач: 1 | В очереди: 76 | Ошибок: 0 | Заблокировано: 0 | Запросов: 2.0 req/sec | Средняя задержка: 0.54 сек
[████

## Тестирование производительности

In [18]:
import copy
import cProfile
import pstats
import tracemalloc

LATENCY = 0.05
FILLER = "<p>" + "Текст страницы для разбора. " * 40 + "</p>"


def make_page(number: int) -> str:
    links = "".join(
        f'<a href="https://perf.test/{child}">стр {child}</a>'
        for child in range(number * 3 + 1, number * 3 + 4)
    )
    return (
        f"<html><head><title>Страница {number}</title></head>"
        f"<body><h1>Страница {number}</h1>{FILLER}{links}</body></html>"
    )


def page_number(url: str) -> int:
    return int(url.rsplit("/", 1)[1])


class SyncCrawler:
    def crawl(self, start_url: str, max_pages: int) -> dict:
        queue = deque([start_url])
        seen = {start_url}
        results = {}
        while queue and len(results) < max_pages:
            url = queue.popleft()
            time.sleep(LATENCY)
            soup = BeautifulSoup(make_page(page_number(url)), "lxml")
            links = [tag["href"] for tag in soup.find_all("a", href=True)]
            results[url] = {"title": soup.title.get_text(strip=True), "links": links}
            for link in links:
                if link not in seen:
                    seen.add(link)
                    queue.append(link)
        return results


class PerfCrawler(AdvancedCrawler):
    async def try_fetch(self, url):
        self.attempts[url] = self.attempts.get(url, 0) + 1
        async with self.semaphores.acquire(url):
            await asyncio.sleep(LATENCY)
        self.responses[url] = {"status_code": 200, "content_type": "text/html"}
        return make_page(page_number(url))

    def print_progress(self, start_time):
        pass


def perf_config(max_concurrent: int = 20) -> dict:
    config = copy.deepcopy(DEFAULT_CONFIG)
    config["crawler"].update(max_concurrent=max_concurrent, max_depth=20)
    config["logging"].update(log_file=None, console_level="WARNING")
    return config


async def run_async(max_pages: int, max_concurrent: int = 20, measure_memory: bool = False):
    crawler = PerfCrawler(perf_config(max_concurrent))
    if measure_memory:
        tracemalloc.start()
    started = time.perf_counter()
    results = await crawler.crawl(["https://perf.test/0"], max_pages=max_pages)
    elapsed = time.perf_counter() - started
    peak = 0
    if measure_memory:
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
    await crawler.close()
    return len(results), elapsed, peak

### Сравнение с синхронным краулером

In [19]:
started = time.perf_counter()
sync_results = SyncCrawler().crawl("https://perf.test/0", 100)
sync_time = time.perf_counter() - started
pages, async_time, _ = await run_async(100)

print(f"Синхронный: {len(sync_results)} стр за {sync_time:.2f} сек")
print(f"Асинхронный: {pages} стр за {async_time:.2f} сек")
print(f"Ускорение: {sync_time / async_time:.1f}x")

Синхронный: 100 стр за 5.44 сек
Асинхронный: 100 стр за 0.48 сек
Ускорение: 11.4x


### Масштабируемость и память

In [20]:
print(f"{'Страниц':>8} {'Время, с':>9} {'Стр/сек':>8} {'Память, МБ':>11}")
for size in (100, 500, 1000):
    pages, elapsed, _ = await run_async(size)
    _, _, peak = await run_async(size, measure_memory=True)
    print(f"{pages:>8} {elapsed:>9.2f} {pages / elapsed:>8.0f} {peak / 1024 / 1024:>11.1f}")

 Страниц  Время, с  Стр/сек  Память, МБ
     100      0.48      210         0.9
     500      1.54      325         2.7
    1000      2.84      352         4.8


### Влияние числа одновременных задач

In [21]:
print(f"{'Задач':>8} {'Время, с':>9} {'Стр/сек':>8}")
for workers in (5, 20, 50):
    pages, elapsed, _ = await run_async(1000, max_concurrent=workers)
    print(f"{workers:>8} {elapsed:>9.2f} {pages / elapsed:>8.0f}")

   Задач  Время, с  Стр/сек
       5     10.85       92
      20      2.84      352
      50      1.33      755


### Поиск узких мест

In [23]:
profiler = cProfile.Profile()
profiler.enable()
await run_async(500)
profiler.disable()
pstats.Stats(profiler).strip_dirs().sort_stats("tottime").print_stats(8);

         1343484 function calls (1335187 primitive calls) in 1.582 seconds

   Ordered by: internal time
   List reduced from 519 to 8 due to restriction <8>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    982/1    1.095    0.001    0.000    0.000 {method 'control' of 'select.kqueue' objects}
  500/353    0.021    0.000    0.077    0.000 _lxml.py:488(feed)
     6000    0.017    0.000    0.068    0.000 filter.py:130(filter)
233776/232772    0.017    0.000    0.022    0.000 {built-in method builtins.isinstance}
     5500    0.016    0.000    0.043    0.000 filter.py:378(__init__)
     6001    0.015    0.000    0.030    0.000 parse.py:501(_urlsplit)
     4500    0.014    0.000    0.054    0.000 _lxml.py:323(start)
     5500    0.013    0.000    0.137    0.000 element.py:1338(_find_all)


